# Chapter 8 — RAG Generation
## Topic 5: Streaming Responses in a RAG Pipeline

**Run cells in order.**

- Module 1: Setup — simulated claim-by-claim streaming events (offline-safe)
- Module 2: Naive full streaming (unverified) vs. claim-level verified streaming
- Module 3: Time-to-first-verified-claim vs. total latency — the actual UX metric
- Module 4: Mid-stream interruption handling

**Install:** none beyond the standard library.


## Module 1: Setup — Simulated Streaming Events

**Requires:** nothing

In [ ]:
"""
MODULE 1: Setup

Simulates a claim-by-claim streaming sequence with artificial per-claim
generation delays and verification outcomes -- offline-safe stand-in for
a real client.messages.stream() tool-use event sequence.
"""

import time

SIMULATED_STREAM_EVENTS = [
    {"claim": "Nominee-initiated withdrawal is exempt from this penalty.",
     "generation_delay_ms": 500, "tier1_flagged": True},   # needs Tier 2 -- generated FIRST
    {"claim": "The penalty for premature withdrawal is 1 percent.",
     "generation_delay_ms": 400, "tier1_flagged": False},
    {"claim": "This applies to all Fixed Deposit tenures.",
     "generation_delay_ms": 350, "tier1_flagged": False},
]

print("Simulated answer will stream as 3 claims:")
for i, e in enumerate(SIMULATED_STREAM_EVENTS, start=1):
    flag = " [WILL BE FLAGGED FOR TIER 2]" if e["tier1_flagged"] else ""
    print(f"  Claim {i} (~{e['generation_delay_ms']}ms to generate): {e['claim']}{flag}")

print("\nModule 1 complete. Run Module 2.")


## Module 2: Naive Streaming vs. Claim-Level Verified Streaming

**Requires:** Module 1

In [ ]:
"""
MODULE 2: Naive vs. Verified Streaming

Naive: shows every claim immediately as it's generated, NO verification gate.
Verified: only shows claims that pass Tier 1 immediately; flagged claims
  are held back and marked "pending" until Tier 2 resolves (simulated here
  as an immediate resolution for clarity -- in production this is async).
"""

def naive_stream(events: list):
    """UNSAFE for a regulated domain -- shown to illustrate what NOT to do."""
    timeline = []
    t = 0
    for e in events:
        t += e["generation_delay_ms"]
        timeline.append({"time_ms": t, "action": "DISPLAYED_UNVERIFIED", "claim": e["claim"]})
    return timeline


def verified_claim_stream(events: list):
    """Claim-level verified streaming: Tier 1 passes display immediately,
    Tier 1 flags are held pending Tier 2."""
    timeline = []
    t = 0
    pending = []
    for e in events:
        t += e["generation_delay_ms"]
        if e["tier1_flagged"]:
            timeline.append({"time_ms": t, "action": "HELD_PENDING_TIER2", "claim": e["claim"]})
            pending.append((t, e["claim"]))
        else:
            timeline.append({"time_ms": t, "action": "DISPLAYED_VERIFIED", "claim": e["claim"]})

    # Simulate Tier 2 resolution arriving shortly after stream completes
    for original_t, claim in pending:
        resolve_t = t + 300  # Tier 2 LLM-judge round-trip, illustrative
        timeline.append({"time_ms": resolve_t, "action": "TIER2_RESOLVED_DISPLAYED", "claim": claim})

    return timeline


print("=" * 65)
print("NAIVE STREAMING (unverified -- NOT recommended for this domain)")
print("=" * 65)
for event in naive_stream(SIMULATED_STREAM_EVENTS):
    print(f"  t={event['time_ms']:>4}ms | {event['action']:<22} | {event['claim'][:50]}")

print("\n" + "=" * 65)
print("CLAIM-LEVEL VERIFIED STREAMING (this project's recommended approach)")
print("=" * 65)
for event in verified_claim_stream(SIMULATED_STREAM_EVENTS):
    print(f"  t={event['time_ms']:>4}ms | {event['action']:<22} | {event['claim'][:50]}")

print("\nModule 2 complete. Run Module 3.")


## Module 3: Time-to-First-Verified-Claim vs. Total Latency

**Requires:** Module 1, Module 2

In [ ]:
"""
MODULE 3: The Actual UX-Relevant Metric

Compares time-to-first-content (naive streaming's headline metric) against
time-to-first-VERIFIED-content and time-to-full-resolution -- the metrics
that actually matter for a regulated domain.
"""

naive_timeline = naive_stream(SIMULATED_STREAM_EVENTS)
verified_timeline = verified_claim_stream(SIMULATED_STREAM_EVENTS)

naive_first_content = naive_timeline[0]["time_ms"]
verified_first_verified = next(e["time_ms"] for e in verified_timeline if e["action"] == "DISPLAYED_VERIFIED")
verified_full_resolution = max(e["time_ms"] for e in verified_timeline)
non_streaming_total = sum(e["generation_delay_ms"] for e in SIMULATED_STREAM_EVENTS) + 300  # + Tier 2

print(f"{'Approach':<35} | {'Metric':<30} | Time")
print("-" * 80)
print(f"{'Naive streaming':<35} | {'Time to FIRST content (any)':<30} | {naive_first_content}ms")
print(f"{'Claim-level verified streaming':<35} | {'Time to first VERIFIED claim':<30} | {verified_first_verified}ms")
print(f"{'Claim-level verified streaming':<35} | {'Time to FULL resolution':<30} | {verified_full_resolution}ms")
print(f"{'No streaming (baseline)':<35} | {'Time to complete answer':<30} | {non_streaming_total}ms")

print(f"""
Interpretation:
  Naive streaming shows SOMETHING fastest ({naive_first_content}ms), but that
    something may be unverified and later wrong.
  Verified streaming shows the FIRST GENUINELY TRUSTWORTHY content at
    {verified_first_verified}ms -- slower than naive, but every displayed
    token is real, and this is still meaningfully faster than waiting for
    full resolution ({verified_full_resolution}ms) or the non-streaming
    baseline ({non_streaming_total}ms).
  This is why "time to first verified claim" -- not "time to first token" --
    is the metric that should be tracked in production for this design.
""")

print("Module 3 complete. Run Module 4.")


## Module 4: Mid-Stream Interruption Handling

**Requires:** Module 1, Module 2

In [ ]:
"""
MODULE 4: Mid-Stream Interruption Handling

Simulates a network failure partway through a stream, and shows the
difference between a graceful partial-answer state (customer sees what
was verified so far, with a clear incomplete-answer indicator) and a
silent, confusing truncation.
"""

def handle_stream_with_interruption(events: list, fail_after_claim: int):
    """Simulates the stream being cut off after a given claim index."""
    delivered = []
    for i, e in enumerate(events):
        if i >= fail_after_claim:
            return {
                "status": "INTERRUPTED",
                "delivered_claims": delivered,
                "message": f"Stream failed after {len(delivered)}/{len(events)} claims delivered.",
            }
        if not e["tier1_flagged"]:
            delivered.append(e["claim"])
    return {"status": "COMPLETE", "delivered_claims": delivered}


result = handle_stream_with_interruption(SIMULATED_STREAM_EVENTS, fail_after_claim=2)

print(f"Stream status: {result['status']}")
print(f"Claims delivered before interruption: {len(result['delivered_claims'])}")
for c in result["delivered_claims"]:
    print(f"  DELIVERED (verified): {c}")

print(f"""
GOOD UX pattern (what the pipeline should do):
  Show the {len(result['delivered_claims'])} claim(s) that WERE verified and
  delivered, with an explicit "connection interrupted, please resend or
  refresh" indicator -- the customer sees real content plus a clear signal
  the answer is incomplete, not a silently truncated response that looks
  complete but isn't.

BAD UX pattern (to avoid):
  Silently stopping with no indicator, leaving the customer to assume the
  partial answer IS the complete answer -- especially dangerous in this
  regulated domain if the missing claims (e.g. an exemption or condition)
  would have changed how the customer should act on the information shown.
""")

print("=" * 65)
print("CHAPTER 8 TOPIC 5 SUMMARY")
print("=" * 65)
print("""
Naive streaming shows unverified content fastest -- not acceptable for
  this project's regulated domain given Topics 2/4's verification guarantees.
Claim-level verified streaming: Tier-1-passed claims display immediately;
  Tier-1-flagged claims are held pending Tier 2, delivered as a follow-up.
The UX-relevant metric is TIME TO FIRST VERIFIED CLAIM, not time to first
  token -- Module 3 made this distinction concrete with real numbers.
Mid-stream interruptions must be handled with an explicit incomplete-answer
  indicator, never a silent truncation.
Given this project's typically short answers (31-word average emails),
  validate with real latency data whether this complexity is worth building
  before committing engineering effort.

Next: Topic 6 -- Multi-Turn RAG (Conversation History + Retrieval)
""")
